# Modeling — Fraud Detection

Covers:
- Stratified train-test split (already done in feature-engineering.ipynb)
- Baseline: Logistic Regression
- Ensemble: XGBoost with hyperparameter tuning
- Stratified K-Fold cross-validation (k=5)
- Model comparison and selection
- Save best models to `models/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (
    average_precision_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, precision_recall_curve
)

sns.set_theme(style='whitegrid')

## 1. Load Processed Data

In [ ]:
# Fraud_Data
fraud_train = pd.read_csv('../data/processed/fraud_train.csv')
fraud_test  = pd.read_csv('../data/processed/fraud_test.csv')

X_train_f = fraud_train.drop(columns=['class'])
y_train_f = fraud_train['class']
X_test_f  = fraud_test.drop(columns=['class'])
y_test_f  = fraud_test['class']

# CreditCard
cc_train = pd.read_csv('../data/processed/creditcard_train.csv')
cc_test  = pd.read_csv('../data/processed/creditcard_test.csv')

X_train_cc = cc_train.drop(columns=['Class'])
y_train_cc = cc_train['Class']
X_test_cc  = cc_test.drop(columns=['Class'])
y_test_cc  = cc_test['Class']

print('Fraud_Data  — Train:', X_train_f.shape, '| Test:', X_test_f.shape)
print('CreditCard  — Train:', X_train_cc.shape, '| Test:', X_test_cc.shape)

## 2. Evaluation Helper

In [ ]:
def evaluate(model, X_test, y_test, label=''):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc_pr = average_precision_score(y_test, y_proba)
    roc    = roc_auc_score(y_test, y_proba)
    f1     = f1_score(y_test, y_pred)

    print(f'--- {label} ---')
    print(f'AUC-PR : {auc_pr:.4f}')
    print(f'ROC-AUC: {roc:.4f}')
    print(f'F1     : {f1:.4f}')

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud']).plot(cmap='Blues')
    plt.title(f'Confusion Matrix — {label}')
    plt.show()

    # Precision-Recall curve
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(rec, prec)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'PR Curve — {label} (AUC={auc_pr:.3f})')
    plt.show()

    return {'model': label, 'AUC-PR': auc_pr, 'ROC-AUC': roc, 'F1': f1}

## 3. Baseline — Logistic Regression

In [ ]:
lr_f = LogisticRegression(max_iter=1000, random_state=42)
lr_f.fit(X_train_f, y_train_f)
res_lr_f = evaluate(lr_f, X_test_f, y_test_f, 'LR — Fraud_Data')

lr_cc = LogisticRegression(max_iter=1000, random_state=42)
lr_cc.fit(X_train_cc, y_train_cc)
res_lr_cc = evaluate(lr_cc, X_test_cc, y_test_cc, 'LR — CreditCard')

## 4. Ensemble — XGBoost with Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [3, 5],
    'learning_rate': [0.05, 0.1]
}

# Fraud_Data
xgb_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                          random_state=42, n_jobs=-1)
gs_f = GridSearchCV(xgb_base, param_grid, scoring='average_precision',
                    cv=StratifiedKFold(3), n_jobs=-1, verbose=0)
gs_f.fit(X_train_f, y_train_f)
print('Best params (Fraud_Data):', gs_f.best_params_)
xgb_f = gs_f.best_estimator_
res_xgb_f = evaluate(xgb_f, X_test_f, y_test_f, 'XGBoost — Fraud_Data')

In [ ]:
# CreditCard
gs_cc = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                   random_state=42, n_jobs=-1),
                     param_grid, scoring='average_precision',
                     cv=StratifiedKFold(3), n_jobs=-1, verbose=0)
gs_cc.fit(X_train_cc, y_train_cc)
print('Best params (CreditCard):', gs_cc.best_params_)
xgb_cc = gs_cc.best_estimator_
res_xgb_cc = evaluate(xgb_cc, X_test_cc, y_test_cc, 'XGBoost — CreditCard')

## 5. Stratified K-Fold Cross-Validation (k=5)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scoring = ['average_precision', 'f1', 'roc_auc']

def cv_report(model, X, y, label):
    cv = cross_validate(model, X, y, cv=skf, scoring=cv_scoring, n_jobs=-1)
    print(f'\n--- CV: {label} ---')
    for s in cv_scoring:
        scores = cv[f'test_{s}']
        print(f'{s:25s}: {scores.mean():.4f} ± {scores.std():.4f}')
    return {
        'model': label,
        'CV AUC-PR mean': cv['test_average_precision'].mean(),
        'CV AUC-PR std':  cv['test_average_precision'].std(),
        'CV F1 mean':     cv['test_f1'].mean(),
        'CV F1 std':      cv['test_f1'].std(),
        'CV ROC-AUC mean': cv['test_roc_auc'].mean(),
        'CV ROC-AUC std':  cv['test_roc_auc'].std(),
    }

# Run on full (pre-SMOTE) training data for unbiased CV
fraud_full = pd.read_csv('../data/processed/fraud_with_country.csv',
                         parse_dates=['signup_time', 'purchase_time'])

# Rebuild features for CV (mirrors feature-engineering.ipynb)
fraud_full['hour_of_day']       = fraud_full['purchase_time'].dt.hour
fraud_full['day_of_week']       = fraud_full['purchase_time'].dt.dayofweek
fraud_full['time_since_signup'] = (fraud_full['purchase_time'] - fraud_full['signup_time']).dt.total_seconds() / 3600
fraud_full['user_tx_count']     = fraud_full.groupby('user_id')['user_id'].transform('count')
fraud_full = fraud_full.sort_values(['user_id', 'purchase_time'])
fraud_full['tx_last_24h']       = fraud_full.groupby('user_id')['purchase_time'].transform(lambda x: x.expanding().count() - 1)

drop_cols = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'ip_int']
cat_cols  = ['source', 'browser', 'sex', 'country']
fraud_full = fraud_full.drop(columns=drop_cols, errors='ignore')
fraud_full = pd.get_dummies(fraud_full, columns=cat_cols, drop_first=True)

X_cv_f = fraud_full.drop(columns=['class'])
y_cv_f = fraud_full['class']

cv_lr_f   = cv_report(LogisticRegression(max_iter=1000, random_state=42), X_cv_f, y_cv_f, 'LR — Fraud_Data')
cv_xgb_f  = cv_report(XGBClassifier(**gs_f.best_params_, use_label_encoder=False,
                                     eval_metric='logloss', random_state=42, n_jobs=-1),
                       X_cv_f, y_cv_f, 'XGBoost — Fraud_Data')

In [ ]:
# CreditCard CV
cc_full = pd.read_csv('../data/processed/creditcard_cleaned.csv')
X_cv_cc = cc_full.drop(columns=['Class'])
y_cv_cc = cc_full['Class']

cv_lr_cc  = cv_report(LogisticRegression(max_iter=1000, random_state=42), X_cv_cc, y_cv_cc, 'LR — CreditCard')
cv_xgb_cc = cv_report(XGBClassifier(**gs_cc.best_params_, use_label_encoder=False,
                                     eval_metric='logloss', random_state=42, n_jobs=-1),
                       X_cv_cc, y_cv_cc, 'XGBoost — CreditCard')

## 6. Model Comparison

In [ ]:
# Hold-out test set comparison
results_f  = pd.DataFrame([res_lr_f,  res_xgb_f])
results_cc = pd.DataFrame([res_lr_cc, res_xgb_cc])

print('=== Fraud_Data — Test Set ===')
print(results_f.set_index('model').to_string())

print('\n=== CreditCard — Test Set ===')
print(results_cc.set_index('model').to_string())

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in zip(axes, [results_f, results_cc], ['Fraud_Data', 'CreditCard']):
    df.set_index('model')[['AUC-PR', 'ROC-AUC', 'F1']].plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# CV comparison table
cv_results = pd.DataFrame([cv_lr_f, cv_xgb_f, cv_lr_cc, cv_xgb_cc])
print('=== Cross-Validation Summary ===')
print(cv_results.set_index('model').to_string())

## 7. Model Selection

**Selected model: XGBoost**

XGBoost consistently outperforms Logistic Regression on AUC-PR and F1 across both datasets and in cross-validation. 
- AUC-PR is the primary metric because it is robust to class imbalance and captures the precision-recall tradeoff directly relevant to fraud detection.
- XGBoost handles non-linear feature interactions (e.g., `time_since_signup` × `user_tx_count`) that Logistic Regression cannot capture without manual feature crosses.
- Logistic Regression is retained as an interpretable baseline for comparison and regulatory explainability.

Both models are saved to `models/` for use in SHAP explainability (Task 3).

In [ ]:
joblib.dump(xgb_f,  '../models/xgb_fraud.pkl')
joblib.dump(xgb_cc, '../models/xgb_creditcard.pkl')
joblib.dump(lr_f,   '../models/lr_fraud.pkl')
joblib.dump(lr_cc,  '../models/lr_creditcard.pkl')
print('Models saved to models/')